<a href="https://colab.research.google.com/github/AhmedE16/flyrank-ai-intern-ahmed-essam/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
print("""
Task type: Scoring / Ranking, built on top of a binary classification model.

The end output a reviewer needs is a ranked list - "review these pages first" - not
a single yes/no label. But the ranking itself is produced by a classifier: a model
that estimates the probability a page is declining (or will decline), and I sort
pages by that probability. This mirrors exactly what Notebook 01/02 did - logistic
regression, decision tree, and random forest all output a probability, which becomes
the ranking signal, scored with Precision@K rather than plain accuracy.

So: classification is the mechanism, ranking is the actual task the business needs.
""")


Task type: Scoring / Ranking, built on top of a binary classification model.

The end output a reviewer needs is a ranked list - "review these pages first" - not
a single yes/no label. But the ranking itself is produced by a classifier: a model
that estimates the probability a page is declining (or will decline), and I sort
pages by that probability. This mirrors exactly what Notebook 01/02 did - logistic
regression, decision tree, and random forest all output a probability, which becomes
the ranking signal, scored with Precision@K rather than plain accuracy.

So: classification is the mechanism, ranking is the actual task the business needs.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("""
Target (starting point, a proxy): is_declining_label = (trend_direction == "down")

This is the starter dataset's built-in label. It's a proxy, not a true future outcome -
it's calculated from the current window, not something that happened after a decision
point. I'm using it now because it's what the starter data supports, but per the lane
guide I want to move toward a stronger label once I have warehouse access in Week 3+:

  prior 90 days of features -> decline over next 30 days

That version is a genuine future outcome, which removes the risk of the label secretly
encoding information from the same window as my features (leakage).
""")

print(df["is_declining_label"].value_counts())
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.

Target (starting point, a proxy): is_declining_label = (trend_direction == "down")

This is the starter dataset's built-in label. It's a proxy, not a true future outcome -
it's calculated from the current window, not something that happened after a decision
point. I'm using it now because it's what the starter data supports, but per the lane
guide I want to move toward a stronger label once I have warehouse access in Week 3+:

  prior 90 days of features -> decline over next 30 days

That version is a genuine future outcome, which removes the risk of the label secretly
encoding information from the same window as my features (leakage).

is_declining_label
1    16262
0    13738
Name: count, dtype: int64
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
print("""
Metric: Precision@50 (extendable to Precision@20).

I'm not using plain accuracy, because the real decision isn't "classify every page
correctly" - it's "which pages does a reviewer with limited capacity look at first?"
Precision@50 asks: of the top 50 pages my ranking puts first, how many are actually
declining? That directly matches how the output gets used.

Baseline to beat, from Notebook 01: hand-written rule Precision@50 = 0.240.
"Good" means clearing that baseline by a meaningful margin - the random forest in the
starter pipeline already reaches 0.740, so that's my rough ceiling to aim toward, not
necessarily match exactly, once I move to a stricter future-window label.
""")


Metric: Precision@50 (extendable to Precision@20).

I'm not using plain accuracy, because the real decision isn't "classify every page
correctly" - it's "which pages does a reviewer with limited capacity look at first?"
Precision@50 asks: of the top 50 pages my ranking puts first, how many are actually
declining? That directly matches how the output gets used.

Baseline to beat, from Notebook 01: hand-written rule Precision@50 = 0.240.
"Good" means clearing that baseline by a meaningful margin - the random forest in the
starter pipeline already reaches 0.740, so that's my rough ceiling to aim toward, not
necessarily match exactly, once I move to a stricter future-window label.



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
cols = ["content_id", "client_id", "trend_direction", "is_declining_label",
        "impressions_90d", "sessions_90d", "avg_position", "ctr",
        "word_count", "content_age_days", "days_since_last_update"]

lane_slice = df[cols].copy()
print(f"Shape: {lane_slice.shape[0]:,} rows, {lane_slice.shape[1]} columns")
print("One row = one page (content_id), matching the starter dataset's grain.\n")
print(lane_slice.head(10))

print("""
Unit of analysis: one page. Each row is a single content item (content_id), with its
observed 90-day signals and its label. This matches the decision grain too - a reviewer
opens and reviews one page at a time, so scoring at the page level is the right unit,
not client-level or query-level.
""")

Shape: 30,000 rows, 11 columns
One row = one page (content_id), matching the starter dataset's grain.

             content_id          client_id trend_direction  \
0  content_304f48230142  client_f369cb89fc            down   
1  content_a1fb4e703a9e  client_4e07408562            down   
2  content_9aa793d4d895  client_7f2253d7e2            down   
3  content_331d6c4de07b  client_19581e27de          stable   
4  content_d99b7a2d90ca  client_3fdba35f04            down   
5  content_d4084a4bc775  client_f369cb89fc            down   
6  content_9a34b442b552  client_8722616204            down   
7  content_a63219c6e95a  client_19581e27de          stable   
8  content_5e6c160719bc  client_6208ef0f77            down   
9  content_c27558df2b0c  client_19581e27de            down   

   is_declining_label  impressions_90d  sessions_90d  avg_position   ctr  \
0                   1             3803            17          10.6  0.76   
1                   1            15320             9          

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
print("""
A fixed rule like "flag if days_since_last_update >= 180 and impressions_90d >= 500"
can only combine a couple of thresholds with AND/OR logic. But declining pages don't
follow one clean pattern - a page can be declining because of position loss, CTR
collapse, thin content, or staleness, often in combination, and the right combination
differs by content type and position tier. A hand-written rule has to pick one story
and apply it uniformly.

The starter pipeline already shows this isn't just theory: the hand-written baseline
gets Precision@50 = 0.240, while a random forest - which can learn nonlinear
interactions between many signals at once - reaches 0.740, roughly 3x better at the
exact same task. That gap is the evidence that the real pattern is too messy for a
short if-statement, and that learning the pattern from data captures something a
fixed rule structurally can't.
""")


A fixed rule like "flag if days_since_last_update >= 180 and impressions_90d >= 500"
can only combine a couple of thresholds with AND/OR logic. But declining pages don't
follow one clean pattern - a page can be declining because of position loss, CTR
collapse, thin content, or staleness, often in combination, and the right combination
differs by content type and position tier. A hand-written rule has to pick one story
and apply it uniformly.

The starter pipeline already shows this isn't just theory: the hand-written baseline
gets Precision@50 = 0.240, while a random forest - which can learn nonlinear
interactions between many signals at once - reaches 0.740, roughly 3x better at the
exact same task. That gap is the evidence that the real pattern is too messy for a
short if-statement, and that learning the pattern from data captures something a
fixed rule structurally can't.



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.